# Temporary Ablation Study

This notebook reruns ablation evaluation every time you run the evaluation cell.

Scope for this version:
- Uses only Applied questions
- Uses only Attention + MemGPT document IDs
- Saves fresh results to `playground/temporary_albation_results.json`

In [3]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'backend').exists():
            return candidate
    raise RuntimeError('Could not locate repository root')

repo_root = find_repo_root(Path.cwd())
backend_dir = repo_root / 'backend'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

repo_root

PosixPath('/home/aman/storage/Python/Projects/Research Paper Assistant')

In [4]:
import json
from collections import Counter

from backend.evaluation import evaluate_ablation as ablation
from backend.evaluation.config import PAPERS

# Force outputs for this temporary notebook into playground
ablation.RESULTS_OUTPUT = repo_root / 'playground' / 'temporary_albation_results.json'

# Restrict ablation to Applied questions from Attention + MemGPT only
paper_map = {p['paper_id']: p for p in PAPERS}
target_doc_ids = {
    paper_map['paper_applied']['document_id'],  # Attention paper
    paper_map['paper_memgpt']['document_id'],   # MemGPT paper
}

bm25_status = {
    doc_id: (repo_root / 'output' / f'{doc_id}_bm25.pkl').exists()
    for doc_id in sorted(target_doc_ids)
}

entries = ablation._load_dataset(ablation.DATASET_PATH)
filtered_entries = [
    e for e in entries
    if e.get('paper_type') == 'Applied' and e.get('document_id') in target_doc_ids
]

if not filtered_entries:
    raise RuntimeError('No Applied entries found for Attention + MemGPT in qa_pairs.json')

print('Running ablation for Applied subset only (Attention + MemGPT).')
print(f"Output will be saved to: {ablation.RESULTS_OUTPUT}")
print(f"Total filtered questions: {len(filtered_entries)}")
print(f"Questions by document_id: {dict(Counter(e.get('document_id') for e in filtered_entries))}")
print(f"BM25 availability by document_id: {bm25_status}")

pipeline = ablation.RetrievalPipeline()
config_results = {}
all_failures = {}

for cfg in ablation.CONFIGURATIONS:
    config_name = cfg['name']
    print(f"\nTesting: {config_name}")

    results_for_config = []
    failed_questions = []

    for idx, entry in enumerate(filtered_entries, 1):
        try:
            result = ablation._evaluate_entry(pipeline, cfg, entry)
            results_for_config.append(result)
        except Exception as exc:
            failed_questions.append({'question': entry.get('question', ''), 'error': str(exc)})

        if idx % 5 == 0:
            print(f"  [{idx}/{len(filtered_entries)}] evaluated...")

    if not results_for_config:
        raise RuntimeError(f'No successful evaluations for configuration: {config_name}')

    mean_p3 = sum(r['precision_at_3'] for r in results_for_config) / len(results_for_config)
    mean_p5 = sum(r['precision_at_5'] for r in results_for_config) / len(results_for_config)
    mean_r5 = sum(r['recall_at_5'] for r in results_for_config) / len(results_for_config)
    mean_mrr = sum(r['reciprocal_rank'] for r in results_for_config) / len(results_for_config)

    config_results[config_name] = {
        'precision_at_3': round(mean_p3, 2),
        'precision_at_5': round(mean_p5, 2),
        'recall_at_5': round(mean_r5, 2),
        'reciprocal_rank': round(mean_mrr, 2),
        'num_evaluated': len(results_for_config),
        'num_failed': len(failed_questions),
        'detailed_results': results_for_config,
    }
    all_failures[config_name] = failed_questions

    print(
        f"  Precision@3={mean_p3:.2f}, Precision@5={mean_p5:.2f}, "
        f"Recall@5={mean_r5:.2f}, MRR={mean_mrr:.2f}"
    )
    if failed_questions:
        print(f"  Failed questions: {len(failed_questions)}")

baseline = ablation.CONFIGURATIONS[0]['name']
full = ablation.CONFIGURATIONS[-1]['name']
improvement_p5 = config_results[full]['precision_at_5'] - config_results[baseline]['precision_at_5']
improvement_mrr = config_results[full]['reciprocal_rank'] - config_results[baseline]['reciprocal_rank']

output_data = {
    'summary': {
        'scope': {
            'paper_type': 'Applied',
            'document_ids': sorted(target_doc_ids),
            'question_count': len(filtered_entries),
            'bm25_available': bm25_status,
        },
        'configurations': [
            {
                'name': cfg['name'],
                'description': cfg['description'],
                'precision_at_3': config_results[cfg['name']]['precision_at_3'],
                'precision_at_5': config_results[cfg['name']]['precision_at_5'],
                'recall_at_5': config_results[cfg['name']]['recall_at_5'],
                'reciprocal_rank': config_results[cfg['name']]['reciprocal_rank'],
                'num_questions': config_results[cfg['name']]['num_evaluated'],
                'num_failed': config_results[cfg['name']]['num_failed'],
            }
            for cfg in ablation.CONFIGURATIONS
        ],
        'improvement': {
            'precision_at_5': round(improvement_p5, 2),
            'reciprocal_rank': round(improvement_mrr, 2),
        },
    },
    'failures': all_failures,
    'detailed_results': config_results,
}

ablation.RESULTS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ablation.RESULTS_OUTPUT.write_text(json.dumps(output_data, indent=2), encoding='utf-8')

data = output_data
summary = data.get('summary', {})
summary.get('configurations', [])

INFO:rag.retrieval.pipeline:RetrievalPipeline: no BM25 encoder for document_id=None; using dense-only retrieval


Running ablation for Applied subset only (Attention + MemGPT).
Output will be saved to: /home/aman/storage/Python/Projects/Research Paper Assistant/playground/temporary_albation_results.json
Total filtered questions: 17
Questions by document_id: {'bd077a96-5a38-5281-993e-10cf869afcde': 9, 'a85211d3-4ad5-52d0-9a49-7d351a58fe31': 8}
BM25 availability by document_id: {'a85211d3-4ad5-52d0-9a49-7d351a58fe31': True, 'bd077a96-5a38-5281-993e-10cf869afcde': True}

Testing: Dense Only


INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO:rag.retrieval.indexing.qdrant_store:QdrantStoreManager: connected to cloud at https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:rag.retrieval.embeddings.dense_encoder:DenseEncoder: cold-loading model BAAI/bge-small-en-v1.5 ...
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO:rag.retrieval.embeddings.dense_encoder:DenseEncoder: model loaded in 5.76s
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: P

  [5/17] evaluated...


INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/query "HTTP/1.1 200 OK"
INFO:rag.retrieval.search.hybrid_retriever:HybridRetriever: query='What overall architecture does the Transformer use for both ' → 5 results (doc_filter=None)
INFO:rag.retrieval.pipeline:RetrievalPipeline: no BM25 encoder for document_id=None; using dense-only retrieval
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cl

...

INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/query "HTTP/1.1 200 OK"
INFO:rag.retrieval.search.hybrid_retriever:HybridRetriever: query='What accuracy improvement does adding MemGPT provide to GPT-' → 5 results (doc_filter=a85211d3-4ad5-52d0-9a49-7d351a58fe31)
INFO:rag.retrieval.search.reranker:FlashRankReranker: reranked 5 → 5 results for query 'What accuracy improvement does adding MemGPT provide to GPT-'
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Reque

  [15/17] evaluated...


INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers/points/query "HTTP/1.1 200 OK"
INFO:rag.retrieval.search.hybrid_retriever:HybridRetriever: query='What are the three components of MemGPT's main context and h' → 5 results (doc_filter=a85211d3-4ad5-52d0-9a49-7d351a58fe31)
INFO:rag.retrieval.search.reranker:FlashRankReranker: reranked 5 → 5 results for query 'What are the three components of MemGPT's main context and h'
INFO:httpx:HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Reque

  Precision@3=0.29, Precision@5=0.21, Recall@5=0.59, MRR=0.54


[{'name': 'Dense Only',
  'description': 'Dense vector retrieval, no BM25, no reranker, no section filter',
  'precision_at_3': 0.24,
  'precision_at_5': 0.14,
  'recall_at_5': 0.41,
  'reciprocal_rank': 0.34,
  'num_questions': 17,
  'num_failed': 0},
 {'name': 'Dense + BM25 (no reranker)',
  'description': 'Hybrid retrieval, no reranker, no section filter',
  'precision_at_3': 0.24,
  'precision_at_5': 0.15,
  'recall_at_5': 0.44,
  'reciprocal_rank': 0.42,
  'num_questions': 17,
  'num_failed': 0},
 {'name': 'Full System',
  'description': 'Hybrid + reranker + section-scoped filter',
  'precision_at_3': 0.29,
  'precision_at_5': 0.21,
  'recall_at_5': 0.59,
  'reciprocal_rank': 0.54,
  'num_questions': 17,
  'num_failed': 0}]

In [5]:
import pandas as pd

scope = summary.get('scope', {})
print('Scope:')
print(f"  paper_type: {scope.get('paper_type', 'N/A')}")
print(f"  document_ids: {scope.get('document_ids', [])}")
print(f"  question_count: {scope.get('question_count', 'N/A')}")
print(f"  bm25_available: {scope.get('bm25_available', {})}")

configs = summary.get('configurations', [])
results_df = pd.DataFrame([
    {
        'Configuration': c['name'],
        'P@3': c['precision_at_3'],
        'P@5': c['precision_at_5'],
        'R@5': c['recall_at_5'],
        'MRR': c['reciprocal_rank'],
        'Questions': c['num_questions'],
        'Failed': c.get('num_failed', 0),
    }
    for c in configs
])

print('\nTemporary Ablation Study Results:')
print('=' * 80)
print(results_df.to_string(index=False))

improvement = summary.get('improvement', {})
print('\n' + '=' * 80)
print('Improvement (Dense Only -> Full System):')
print(f"  Precision@5: +{improvement.get('precision_at_5', 0)}")
print(f"  Mean Reciprocal Rank: +{improvement.get('reciprocal_rank', 0)}")
print('=' * 80)

Scope:
  paper_type: Applied
  document_ids: ['a85211d3-4ad5-52d0-9a49-7d351a58fe31', 'bd077a96-5a38-5281-993e-10cf869afcde']
  question_count: 17
  bm25_available: {'a85211d3-4ad5-52d0-9a49-7d351a58fe31': True, 'bd077a96-5a38-5281-993e-10cf869afcde': True}

Temporary Ablation Study Results:
             Configuration  P@3  P@5  R@5  MRR  Questions  Failed
                Dense Only 0.24 0.14 0.41 0.34         17       0
Dense + BM25 (no reranker) 0.24 0.15 0.44 0.42         17       0
               Full System 0.29 0.21 0.59 0.54         17       0

Improvement (Dense Only -> Full System):
  Precision@5: +0.07
  Mean Reciprocal Rank: +0.2


In [ ]:
import pandas as pd

# Dataset 1 - Applied Questions (from LaTeX data)
data1 = {
    'Configuration': [
        'Baseline (dense only)',
        'Hybrid BM25 + dense (no reranker)',
        'Full system (hybrid + reranker + section filter)'
    ],
    'Precision@5': [0.18, 0.26, 0.33],
    'Recall@5': [0.65, 0.70, 0.9375],
    'MRR': [0.38, 0.49, 0.59]
}

# Dataset 2 - Additional benchmark
data2 = {
    'Configuration': [
        'Baseline (dense only)',
        'Hybrid BM25 + dense (no reranker)',
        'Full system (hybrid + reranker + section filter)'
    ],
    'Precision@5': [0.42, 0.45, 0.63],
    'Recall@5': [0.53, 0.57, 0.76],
    'MRR': [0.43, 0.53, 0.68]
}

df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)

print('\nAblation Study Results')
print('\nAblation Study Results - Dataset 1 (Applied Questions):')
print('=' * 90)
print(df1.to_string(index=False))

improvement1_p5 = data1['Precision@5'][-1] - data1['Precision@5'][0]
improvement1_mrr = data1['MRR'][-1] - data1['MRR'][0]
print('\n' + '=' * 90)
print('Improvement (Baseline → Full System):')
print(f"  Precision@5: +{improvement1_p5:.2f}")
print(f"  Mean Reciprocal Rank: +{improvement1_mrr:.2f}")

print('\n\nAblation Study Results')
print('\n\nAblation Study Results - Dataset 2:')
print('=' * 90)
print(df2.to_string(index=False))

improvement2_p5 = data2['Precision@5'][-1] - data2['Precision@5'][0]
improvement2_mrr = data2['MRR'][-1] - data2['MRR'][0]
print('\n' + '=' * 90)
print('Improvement (Baseline → Full System):')
print(f"  Precision@5: +{improvement2_p5:.2f}")
print(f"  Mean Reciprocal Rank: +{improvement2_mrr:.2f}")
print('=' * 90)
